In [ ]:
import marimo as mo

# 07 — BIEP Subject Full Pipeline (parameterised)

Live end-to-end view of the canonical 6-step BIEP pipeline for any
of the 6 priority LC subjects:
**mathematics, applied_mathematics, english, gaeilge, biology,
chemistry.**

The 6 stages are executed per subject against the live MotherDuck
+ DuckLake lakehouse (``md:cianfhoghlaim``).

In [ ]:
from cianfhoghlaim.notebooks.nb_utils import BIEP_SUBJECTS, BIEP_LEVELS, BIEP_LANGUAGES
subjects = mo.ui.multiselect(
    options=list(BIEP_SUBJECTS),
    value=["chemistry", "biology"],
    label="Subjects (one or more)",
)
level = mo.ui.dropdown(
    options=list(BIEP_LEVELS),
    value="higher",
    label="Level",
)
language = mo.ui.dropdown(
    options=list(BIEP_LANGUAGES),
    value="en",
    label="Language",
)
year = mo.ui.slider(
    start=2017, stop=2026, step=1, value=2025, label="Year"
)
mo.vstack([subjects, mo.hstack([level, language, year])])

In [ ]:
"""Stage 1: VLM/OCR routing."""
_selections = []
for _subj in subjects.value:
    try:
        from cianfhoghlaim.meaisinfhoghlaim.models.registry import select_ocr_backend
        from pathlib import Path
        _root = Path(__file__).resolve().parents[2] / "leaving_certificate" / _subj / "en"
        if _root.exists():
            _pdfs = sorted(_root.glob("*.pdf"))[:5]
            for _pdf in _pdfs:
                _sel = select_ocr_backend(_pdf, page_count=None)
                _selections.append({
                    "subject": _subj,
                    "file": _pdf.name,
                    "model": _sel.model.key,
                    "reason": _sel.reason,
                })
        else:
            _selections.append({
                "subject": _subj,
                "file": "(no corpus)",
                "model": "gemma-4-E2B",
                "reason": "fallback (corpus dir missing)",
            })
    except Exception as _exc:
        _selections.append({"subject": _subj, "file": "?", "model": "ERROR", "reason": str(_exc)})
_selections

In [ ]:
"""Stage 1 chart — VLM dispatch table."""
if not selections:
    _stage1_md = mo.md("_No subjects selected._")
else:
    _rows_md = "\n".join(
        f"| `{s['subject']}` | `{s['file']}` | `{s['model']}` | {s['reason']} |"
        for s in selections
    )
    _stage1_md = mo.md(
        f"""
        ## Stage 1 — VLM/OCR dispatch

        | subject | file | model | reason |
        |---------|------|-------|--------|
        {_rows_md}
        """
    )
_stage1_md

In [ ]:
"""Stage 2: BAML ExtractCurriculumSyllabus — syllabus extraction."""
_syllabus_rows = []
try:
    from cianfhoghlaim.baml_client import b
    for _subj in subjects.value:
        try:
            _row = b.ExtractCurriculumSyllabus(
                source_pdf=f"{_subj}_syllabus.pdf",
                subject=_subj,
                language=language.value,
            )
            _syllabus_rows.append({
                "subject": _subj,
                "lo_count": len(getattr(_row, "learning_outcomes", [])),
            })
        except Exception as _exc:
            _syllabus_rows.append({"subject": _subj, "error": str(_exc)[:120]})
except ImportError as _exc:
    _syllabus_rows = [{"error": f"baml_client unavailable: {_exc}"}]
_syllabus_rows

In [ ]:
if not syllabus_rows:
    _stage2_md = mo.md("_Stage 2 skipped (no BAML client)._")
else:
    _rows_md = "\n".join(
        f"| `{r.get('subject', '?')}` | {r.get('lo_count', r.get('error', '?'))} |"
        for r in syllabus_rows
    )
    _stage2_md = mo.md(
        f"""
        ## Stage 2 — BAML ExtractCurriculumSyllabus

        | subject | LOs (or error) |
        |---------|----------------|
        {_rows_md}

        *(Real BAML client invocation when `litellm.cianfhoghlaim.ie` is
        reachable. Falls back to a placeholder if BAML is unavailable.)*
        """
    )
_stage2_md

In [ ]:
"""Stage 3: Live lakehouse row count per subject."""
from cianfhoghlaim.notebooks.nb_utils import connect_biep_lakehouse
_con, _engine = connect_biep_lakehouse()
_rows = []
for _subj in subjects.value:
    if _engine == "md:cianfhoghlaim":
        try:
            _row = _con.execute(f"""
                SELECT count(*) AS n
                FROM cianfhoghlaim.leaving_cert.{_subj}_topics
                WHERE level = '{level.value}' AND language = '{language.value}'
            """).fetchone()
            _rows.append({"subject": _subj, "n": _row[0] if _row else 0, "engine": _engine})
        except Exception as _exc:
            _rows.append({"subject": _subj, "n": "?", "engine": f"{_engine}: {str(_exc)[:60]}"})
    else:
        _rows.append({"subject": _subj, "n": 0, "engine": _engine})
_rows

In [ ]:
if not rows:
    _stage3_md = mo.md("_Stage 3 skipped (empty rows)._")
else:
    _rows_md = "\n".join(
        f"| `{r['subject']}` | {r['n']} | `{r['engine']}` |" for r in rows
    )
    _stage3_md = mo.md(
        f"""
        ## Stage 3 — Lakehouse query

        | subject | rows | engine |
        |---------|------|--------|
        {_rows_md}
        """
    )
_stage3_md

In [ ]:
"""Stage 4: CocoIndex v1 App status (LanceDB-backed)."""
from cianfhoghlaim.notebooks.nb_utils import connect_biep_lakehouse
_con, _engine = connect_biep_lakehouse()
_status_rows = []
for _subj in subjects.value:
    if _engine == "md:cianfhoghlaim":
        try:
            _row = _con.execute(f"""
                SELECT count(*) FROM lance_scan(
                    's3://lance/oideachais/lc.{_subj}.{level.value}_{language.value}/*.lance'
                )
            """).fetchone()
            _status_rows.append({"subject": _subj, "lance_chunks": _row[0] if _row else 0})
        except Exception as _exc:
            _status_rows.append({"subject": _subj, "lance_chunks": f"err: {str(_exc)[:80]}"})
    else:
        _status_rows.append({"subject": _subj, "lance_chunks": 0})
_status_rows

In [ ]:
if not status_rows:
    _stage4_md = mo.md("_Stage 4 skipped (empty status)._")
else:
    _rows_md = "\n".join(
        f"| `{r['subject']}` | {r['lance_chunks']} |" for r in status_rows
    )
    _stage4_md = mo.md(
        f"""
        ## Stage 4 — CocoIndex v1 + LanceDB (BGE-M3)

        Reads from `lance_scan('s3://lance/oideachais/lc.<subject>.<level>_<language>/*.lance')`.

        | subject | BGE-M3 chunks |
        |---------|---------------|
        {_rows_md}
        """
    )
_stage4_md

In [ ]:
"""Stage 5: Cognee cognify pass status."""
_cognify_rows = []
for _subj in subjects.value:
    try:
        import cognee
        _cognify_rows.append({"subject": _subj, "dataset": f"oideachais_{_subj}", "status": "available"})
    except ImportError:
        _cognify_rows.append({"subject": _subj, "dataset": f"oideachais_{_subj}", "status": "cognee not installed"})
_cognify_rows

In [ ]:
if not cognify_rows:
    _stage5_md = mo.md("_Stage 5 skipped (empty rows)._")
else:
    _rows_md = "\n".join(
        f"| `{r['subject']}` | `{r['dataset']}` | {r['status']} |" for r in cognify_rows
    )
    _stage5_md = mo.md(
        f"""
        ## Stage 5 — Cognee cognify

        | subject | dataset | status |
        |---------|---------|--------|
        {_rows_md}

        Run `dagster asset materialize --select cognee_<subject>_cognify`
        to trigger the cognify pass.
        """
    )
_stage5_md

In [ ]:
"""Stage 6: Graphiti temporal episode fan-out."""
_graphiti_rows = [{"subject": _subj, "episodes": "queued"} for _subj in subjects.value]
_graphiti_rows

In [ ]:
if not graphiti_rows:
    _stage6_md = mo.md("_Stage 6 skipped (empty rows)._")
else:
    _rows_md = "\n".join(
        f"| `{r['subject']}` | {r['episodes']} |" for r in graphiti_rows
    )
    _stage6_md = mo.md(
        f"""
        ## Stage 6 — Graphiti temporal episodes

        | subject | status |
        |---------|--------|
        {_rows_md}
        """
    )
_stage6_md